### Импорты

In [226]:
import time
import random
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.notebook import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mlflow

### Загрузка данных

In [227]:
url ="https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"
df = pd.read_csv(url, parse_dates=["date"], index_col="date")
series = df["OT"].values.astype(np.float64)

In [228]:
fig_1 = go.Figure()
fig_1.add_trace(
    go.Scatter(y=series)
)
fig_1.update_layout(
    title_text = 'Electricity Transformer Temperature (Oil Temperature)',
    xaxis_title='Индекс',
    yaxis_title='Температура'
)
fig_1.show()

### Константы

In [229]:
N = len(series)

TRAIN_PROP = 0.7
VAL_PROP = 0.1

BATCH_SIZE = 128

L = 96
H = 24

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

MLFLOW_TRACING_URI = 'http://localhost:5000/'

PROJECT_DIR = Path.cwd().parent
SRC_DIT = PROJECT_DIR / 'src'
CHECKPOINT_DIR = SRC_DIT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

mlflow.set_tracking_uri(MLFLOW_TRACING_URI)
mlflow.set_experiment('lab_work_6')

<Experiment: artifact_location='file:C:/projects/pstu-machine-learning-models-and-technologies/lab_work_6/src/mlflow/mlartifacts/639574801878929867', creation_time=1778057099103, experiment_id='639574801878929867', last_update_time=1778057099103, lifecycle_stage='active', name='lab_work_6', tags={}, trace_location=None, workspace='default'>

### Метрики

In [230]:
def calculate_metrics(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    return mae, rmse

### Разбиение данных

In [231]:
train_end = int(N * TRAIN_PROP)
val_end = train_end + int(N * VAL_PROP)

train_data = series[:train_end]
val_data = series[train_end:val_end]
test_data = series[val_end:]

In [232]:
l_train, l_val, l_test = len(train_data), len(val_data), len(test_data)
fig_2_data_traces = [
    ('Train', 0, l_train, train_data),
    ('Val', l_train, l_train+l_val, val_data), 
    ('Test', l_train+l_val, l_train+l_val+l_test, test_data)
]

fig_2 = go.Figure()
for name, start_idx, end_idx, data_trace in fig_2_data_traces:
    x = np.arange(start_idx, end_idx)
    fig_2.add_trace(
        go.Scatter(x=x, y=data_trace, name=name)
    )
fig_2.update_layout(
    title_text = 'Electricity Transformer Temperature (Oil Temperature) - Выборки',
    xaxis_title='Индекс',
    yaxis_title='Температура',
    legend_title='Выборки'
)
fig_2.show()

### Нормализация (Минимаксное масштабирование)

In [233]:
scaler = MinMaxScaler()

train_data_std = scaler.fit_transform(train_data.reshape(-1, 1)).flatten()
val_data_std = scaler.transform(val_data.reshape(-1, 1)).flatten()
test_data_std = scaler.transform(test_data.reshape(-1, 1)).flatten()

### Наивное предсказание

In [234]:
def naive(window, H):
    return np.full(H, window[-1])

In [235]:
def roll_mean(window, H):
    return np.full(H, window.mean())

### Dataset и Dataloader

In [236]:
class TSDataset(Dataset):
    def __init__(self, data, L=96, H=24):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.L, self.H = L, H

    def __len__(self):
        return len(self.data) - self.L - self.H + 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.L].unsqueeze(-1)     # (L, 1)
        y = self.data[idx + self.L : idx + self.L + self.H] # (H,)
        return x, y

In [237]:
train_dataset = TSDataset(train_data_std, L, H)
val_dataset = TSDataset(val_data_std, L, H)
test_dataset = TSDataset(test_data_std, L, H)

In [238]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

### Функции

In [239]:
def checkpoint(dir, i_epoch, model, optimizer, **kwargs):
    save_dict = {key: value for key, value in kwargs.items()}
    save_dict['epoch'] = i_epoch + 1
    save_dict['model_state_dict'] = model.state_dict()
    save_dict['optimizer_state_dict'] = optimizer.state_dict()
    folder_path = CHECKPOINT_DIR / dir
    folder_path.mkdir(exist_ok=True)
    checkpoint_path = folder_path / f'epoch_{i_epoch + 1}.pt'
    torch.save(save_dict, checkpoint_path)
    return

In [240]:
def fit_one_epoch(model, train_dataloader, optimizer, criterion):
    model.train()
    losses_epoch = []
    y_true_epoch = []
    y_pred_epoch = []
    for X_batch, y_batch in train_dataloader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        losses_epoch.append(loss.item())
        y_true_epoch.extend(y_batch.cpu().tolist())
        y_pred_epoch.extend(preds.cpu().tolist())
    losses_mean = sum(losses_epoch) / len(losses_epoch)
    y_true_arr = np.array(y_true_epoch)
    y_pred_arr = np.array(y_pred_epoch)
    mae, rmse = calculate_metrics(y_true_arr, y_pred_arr)
    return losses_mean, mae, rmse

In [241]:
def eval_one_epoch(model, val_dataloader, criterion):
    model.eval()
    losses_epoch = []
    y_true_epoch = []
    y_pred_epoch = []
    for X_batch, y_batch in val_dataloader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        with torch.no_grad():
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
        losses_epoch.append(loss.item())
        y_true_epoch.extend(y_batch.cpu().tolist())
        y_pred_epoch.extend(preds.cpu().tolist())
    losses_mean = sum(losses_epoch) / len(losses_epoch)
    y_true_arr = np.array(y_true_epoch)
    y_pred_arr = np.array(y_pred_epoch)
    mae, rmse = calculate_metrics(y_true_arr, y_pred_arr)
    return losses_mean, mae, rmse

In [242]:
def train_model(model, loaders, optimizer, criterion, max_epochs, run_name=None, i_epoch_start=0):
    if not run_name:
        run_name = f'run_{model._get_name()}_{time.time()}'
    with mlflow.start_run(run_name=run_name):
        pbar = tqdm(range(i_epoch_start, max_epochs + i_epoch_start))
        train_losses_all = []
        mlflow.log_param('optimizer', str(optimizer))
        mlflow.log_param('criterion', str(criterion))
        for i_epoch in pbar:
            pbar.set_description(f'(train) Epoch {i_epoch + 1}')
            train_loss, train_mae, train_rmse = fit_one_epoch(model, loaders['train'], optimizer, criterion)
            pbar.set_description(f'(eval) Epoch {i_epoch + 1}')
            val_loss, val_mae, val_rmse = eval_one_epoch(model, loaders['val'], criterion)
            pbar.set_description('Checkpointing')
            mlflow.log_metric('train_loss', train_loss, step=i_epoch)
            mlflow.log_metric('train_mae', train_mae, step=i_epoch)
            mlflow.log_metric('train_rmse', train_rmse, step=i_epoch)
            mlflow.log_metric('val_loss', val_loss, step=i_epoch)
            mlflow.log_metric('val_mae', val_mae, step=i_epoch)
            mlflow.log_metric('val_rmse', val_rmse, step=i_epoch)
            checkpoint(run_name, i_epoch, model, optimizer, 
                    train_loss=train_loss, train_mae=train_mae, train_rmse=train_rmse,
                    val_loss=val_loss, val_mae=val_mae, val_rmse=val_rmse)
            train_losses_all.append(train_loss)
    mlflow.end_run()
    return

In [243]:
def predict(model, loader):
    model.eval()
    all_preds = []
    for X_batch, y_batch in tqdm(loader, desc='Test mode...'):
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        with torch.no_grad():
            preds = model(X_batch)
        all_preds.extend(preds.cpu().tolist())
    return all_preds

In [244]:
def predict_naive(func, loader):
    all_preds = []
    for X_batch, y_batch in tqdm(loader, desc='Naive test mode...'):
        X_batch, y_batch = X_batch.numpy(), y_batch.numpy()
        for i in range(X_batch.shape[0]):
            window = X_batch[i]
            pred = func(window, H)
            all_preds.append(pred.tolist())
    return all_preds

In [245]:
loaders = {'train': train_dataloader, 'val': val_dataloader, 'test': test_dataloader}

### LSTM

In [246]:
# class LSTMForecaster(nn.Module):
#     def __init__(self, hidden=64, layers=2, H=24, dropout=0.2):
#         super().__init__()
#         self.lstm = nn.LSTM(input_size=1, hidden_size=hidden,
#                             num_layers=layers, batch_first=True,
#                             dropout=dropout)
#         self.head = nn.Linear(hidden, H)

#     def forward(self, x):
#         # x: (batch, L, 1)
#         out, (h_n, c_n) = self.lstm(x)      # out: (batch, L, hidden)
#         last_out = out[:, -1, :]            # (batch, hidden)
#         return self.head(last_out)          # (batch, H)
    
class LSTMForecaster(nn.Module):
    def __init__(self, hidden=64, layers=2, H=24, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, layers, batch_first=True,
        dropout=dropout)
        self.head = nn.Linear(hidden, H)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:,-1])


In [247]:
lstm_model = LSTMForecaster(hidden=64, layers=2, H=H, dropout=0.2).to(DEVICE)
lstm_optimizer = AdamW(lstm_model.parameters(), lr=1e-3)
lstm_criterion = nn.MSELoss()
train_model(model=lstm_model, loaders=loaders, optimizer=lstm_optimizer, criterion=lstm_criterion, max_epochs=30, run_name='LSTM_2_shuffle')

  0%|          | 0/30 [00:00<?, ?it/s]

🏃 View run LSTM_2_shuffle at: http://localhost:5000/#/experiments/639574801878929867/runs/86466ec104144ab1af6b1c8bf78dd8dd
🧪 View experiment at: http://localhost:5000/#/experiments/639574801878929867


In [258]:
cpoint_lstm = torch.load(CHECKPOINT_DIR / 'LSTM_2_shuffle' / 'epoch_30.pt', map_location=DEVICE, weights_only=False)
best_lstm_model = LSTMForecaster(hidden=64, layers=2, H=H, dropout=0.2).to(DEVICE)
best_lstm_model.load_state_dict(cpoint_lstm['model_state_dict'])

<All keys matched successfully>

### Transformer

In [249]:
# class TSTransformer(nn.Module):
#     def __init__(self, d_model=64, nhead=4, layers=2, L=96, H=24, dropout=0.1):
#         super().__init__()
#         self.proj = nn.Linear(1, d_model)
#         # positional encoding
#         self.pos_embed = nn.Embedding(L, d_model)
#         enc_layer = nn.TransformerEncoderLayer(
#             d_model=d_model, nhead=nhead,
#             dim_feedforward=256, dropout=dropout,
#             batch_first=True
#         )
#         self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
#         self.head = nn.Linear(d_model * L, H)

#     def forward(self, x):
#         # x: (batch, L, 1)
#         batch_size, L, _ = x.shape
#         x = self.proj(x)                          # (batch, L, d_model)
#         positions = torch.arange(L, device=x.device).unsqueeze(0).expand(batch_size, -1)
#         pos_emb = self.pos_embed(positions)       # (batch, L, d_model)
#         x = x + pos_emb
#         enc_out = self.encoder(x)                 # (batch, L, d_model)
#         enc_out = enc_out.reshape(batch_size, -1) # (batch, L * d_model)
#         return self.head(enc_out)                 # (batch, H)

class TSTransformer(nn.Module):
    def __init__(self, d_model=64, nhead=4, layers=2, L=96, H=24):
        super().__init__()
        self.proj = nn.Linear(1, d_model)
        enc = nn.TransformerEncoderLayer(
        d_model=d_model, nhead=nhead, dim_feedforward=256,
        dropout=0.1, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=layers)
        self.head = nn.Linear(d_model * L, H)
    def forward(self, x):
        x = self.proj(x)
        x = self.encoder(x)
        return self.head(x.flatten(1))

In [250]:
transformer_model = TSTransformer(d_model=64, nhead=4, layers=2, L=L, H=H).to(DEVICE)
transformer_optimizer = AdamW(transformer_model.parameters(), lr=1e-3)
transformer_criterion = nn.MSELoss()
train_model(model=transformer_model, loaders=loaders, optimizer=transformer_optimizer, criterion=transformer_criterion, max_epochs=30, run_name='Transformer_2_shuffle')

  0%|          | 0/30 [00:00<?, ?it/s]

🏃 View run Transformer_2_shuffle at: http://localhost:5000/#/experiments/639574801878929867/runs/33512c9dd5174b84b1bb73b16d15030a
🧪 View experiment at: http://localhost:5000/#/experiments/639574801878929867


In [257]:
cpoint_transformer = torch.load(CHECKPOINT_DIR / 'Transformer_2_shuffle' / 'epoch_28.pt', map_location=DEVICE, weights_only=False)
best_transformer_model = TSTransformer(d_model=64, nhead=4, layers=2, L=L, H=H).to(DEVICE)
best_transformer_model.load_state_dict(cpoint_transformer['model_state_dict'])

<All keys matched successfully>

### Предсказания

In [259]:
naive_test_preds = predict_naive(naive, loaders['test'])
roll_mean_test_preds = predict_naive(roll_mean, loaders['test'])
lstm_test_preds = predict(best_lstm_model, loaders['test'])
transformer_test_preds = predict(best_transformer_model, loaders['test'])

Naive test mode...:   0%|          | 0/27 [00:00<?, ?it/s]

Naive test mode...:   0%|          | 0/27 [00:00<?, ?it/s]

Test mode...:   0%|          | 0/27 [00:00<?, ?it/s]

Test mode...:   0%|          | 0/27 [00:00<?, ?it/s]

In [260]:
all_y_true = []
for _, y_batch in test_dataloader:
    all_y_true.extend(y_batch.cpu().numpy())

naive_mae, naive_rmse = calculate_metrics(all_y_true, np.array(naive_test_preds))
roll_mean_mae, roll_mean_rmse = calculate_metrics(all_y_true, np.array(roll_mean_test_preds))
lstm_mae, lstm_rmse = calculate_metrics(all_y_true, np.array(lstm_test_preds))
transformer_mae, transformer_rmse = calculate_metrics(all_y_true, np.array(transformer_test_preds))

### Визуализация предсказаний на тестовой выборке

In [264]:
N_PREDS = 5

rnd_preds_idx = random.sample(range(len(test_dataset)), 5)

fig_3_preds_traces = [
    ('naive', naive_test_preds, 'orange'),
    ('roll_mean', roll_mean_test_preds, 'purple'),
    ('lstm', lstm_test_preds, 'cyan'),
    ('transformer', transformer_test_preds, 'magenta')
]

fig_3 = make_subplots(
    rows=N_PREDS,
    cols=1,
)

for i, idx in enumerate(rnd_preds_idx, start=1):
    x, y = test_dataset[idx]
    x = scaler.inverse_transform(x.numpy().reshape(-1, 1)).flatten()
    y = scaler.inverse_transform(y.numpy().reshape(-1, 1)).flatten()

    fig_3.add_trace(
        go.Scatter(
            x=np.arange(-L, 0), y=x,
            mode='lines', name=f'L_{i}',
            line=dict(color='blue')
        ), 
        row=i, col=1
    )
    
    fig_3.add_trace(go.Scatter(
        x=np.arange(0, H), y=y,
        mode='lines', name=f'H_{i}',
        line=dict(color='green')
    ), row=i, col=1)
    
    for name, data_trace, color in fig_3_preds_traces:
        data_trace = scaler.inverse_transform(np.array(data_trace[idx]).reshape(-1, 1)).flatten()
        fig_3.add_trace(go.Scatter(
            x=np.arange(0, H), y=data_trace,
            mode='lines', name=f'{name}_{i}',
            line=dict(color=color, dash='dash')
        ), row=i, col=1)

    # Вертикальная разделительная линия
    fig_3.add_vline(x=0, line_dash='dot', line_color='red', 
                  opacity=0.5, row=i, col=1)
# Общие настройки
fig_3.update_layout(
    title=f'Прогнозы моделей на {N_PREDS} случайных тестовых окнах',
    height=250 * N_PREDS,
    width=1400,
    showlegend=True,
)
fig_3.update_yaxes(title_text='Температура')
fig_3.update_xaxes(title_text='Шаг')
fig_3.show()

### Метрики

In [265]:
all_y_true = []
for _, y_batch in test_dataloader:
    all_y_true.extend(y_batch.cpu().numpy())

naive_mae, naive_rmse = calculate_metrics(all_y_true, np.array(naive_test_preds))
roll_mean_mae, roll_mean_rmse = calculate_metrics(all_y_true, np.array(roll_mean_test_preds))
lstm_mae, lstm_rmse = calculate_metrics(all_y_true, np.array(lstm_test_preds))
transformer_mae, transformer_rmse = calculate_metrics(all_y_true, np.array(transformer_test_preds))

In [266]:
z_fig_4 = [[naive_mae, roll_mean_mae, lstm_mae, transformer_mae], [naive_rmse, roll_mean_rmse, lstm_rmse, transformer_rmse]]
x_fig_4 = ['naive', 'roll_mean', 'lstm', 'transformer']
y_fig_4 = ['mae', 'rmse']
fig_4 = go.Figure(data=go.Heatmap(
                   z=z_fig_4,
                   x=x_fig_4,
                   y=y_fig_4,
                   text=np.round(z_fig_4, 5),
                   texttemplate='%{text}',
                   textfont={"size": 12},
                   colorscale = 'YlOrRd',
                   hoverongaps = False,
                   hovertemplate=   '<b>Метрика</b>: %{y}<br>' +
                                    '<b>Метод</b>: %{x}<br>' +
                                    '<b>Оценка</b>: %{z:.5f}<br>' +
                                    '<extra></extra>',))
fig_4.update_layout(
    title={
        'text': 'Тепловая карта оценки методов прогноза на тестовом множестве',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title="Метод",
    yaxis_title="Метрика",
)
fig_4.show()